<a href="https://colab.research.google.com/github/ryukf333/salmonella-risk-prediction/blob/main/Data_curation_%26_organization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inspect json

In [ ]:

# CELL 1: Connect Google Drive and configure the project

from google.colab import drive
from pathlib import Path
import zipfile
import json
import gc
from collections import Counter

import pandas as pd
from IPython.display import display

# 1. Connect Google Drive
drive.mount("/content/drive")

# 2. Define the project folder
PROJECT_DIR = Path(
    "/content/drive/MyDrive/raw_poultry_ml"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
METADATA_DIR = PROJECT_DIR / "metadata"
REPORT_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = REPORT_DIR / "figures"
RESULTS_DIR = REPORT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"

# 3. Create project directories if they do not exist
for folder in [
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    NOTEBOOKS_DIR,
    METADATA_DIR,
    REPORT_DIR,
    FIGURES_DIR,
    RESULTS_DIR,
    MODELS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# 4. Define the four input archives
YEARS = [2022, 2023, 2024, 2025, 2026]

ZIP_PATHS = {
    year: RAW_DIR / (
        f"raw_poultry_sampling_data_fy{year}.zip"
    )
    for year in YEARS
}

# 5. Check the files
print("PROJECT DIRECTORY:", PROJECT_DIR)
print("=" * 60)

missing_files = []

for year, path in ZIP_PATHS.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(
            f"FY{year}: FOUND | "
            f"{path.name} | {size_mb:.2f} MB"
        )
    else:
        missing_files.append(path.name)
        print(f"FY{year}: MISSING | {path.name}")

if missing_files:
    raise FileNotFoundError(
        "\nPlease upload these files to:\n"
        f"{RAW_DIR}\n\n"
        + "\n".join(missing_files)
    )

print("\nSUCCESS: All five ZIP archives are accessible.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT DIRECTORY: /content/drive/MyDrive/raw_poultry_ml
FY2022: FOUND | raw_poultry_sampling_data_fy2022.zip | 3.43 MB
FY2023: FOUND | raw_poultry_sampling_data_fy2023.zip | 3.41 MB
FY2024: FOUND | raw_poultry_sampling_data_fy2024.zip | 2.28 MB
FY2025: FOUND | raw_poultry_sampling_data_fy2025.zip | 2.33 MB
FY2026: FOUND | raw_poultry_sampling_data_fy2026.zip | 1.76 MB

SUCCESS: All five ZIP archives are accessible.


In [ ]:
# CELL 2: Define the inspection function

def inspect_archive(year, zip_path):
    """
    Inspect one poultry sampling JSON archive.
    Does not clean, modify, or merge the original records.
    """

    print(f"\nInspecting FY{year}: {zip_path.name}")

    # Find the JSON file inside the ZIP archive
    with zipfile.ZipFile(zip_path, "r") as archive:

        json_files = [
            name for name in archive.namelist()
            if name.lower().endswith(".json")
        ]

        if len(json_files) != 1:
            raise ValueError(
                f"Expected one JSON file in {zip_path.name}, "
                f"but found: {json_files}"
            )

        json_name = json_files[0]

        # Read the JSON document
        with archive.open(json_name, "r") as file:
            document = json.load(file)

    # The supplied JSON files use a one-element list at the root
    if isinstance(document, list):
        if len(document) != 1:
            raise ValueError(
                f"Unexpected root list length: {len(document)}"
            )
        document = document[0]

    if not isinstance(document, dict):
        raise ValueError("The JSON root must contain an object.")

    metadata = document.get("metadata", {})
    data = document.get("data", {})
    appendix = document.get("appendix", [])

    if not isinstance(data, dict):
        raise ValueError("'data' is not a dictionary.")

    # Save the full metadata, including the data dictionary
    metadata_path = (
        METADATA_DIR / f"metadata_FY{year}.json"
    )

    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(
            metadata,
            file,
            indent=2,
            ensure_ascii=False
        )

    # Summarize useful metadata fields
    metadata_summary = {
        "fiscal_year": year,
        "json_file": json_name,
        "data_product_name": metadata.get("data_product_name"),
        "description": metadata.get("description"),
        "source": metadata.get("source"),
        "creation_date": metadata.get("creation_date"),
        "metadata_keys": "; ".join(metadata.keys()),
        "data_table_names": "; ".join(data.keys()),
        "appendix_entries": len(appendix)
        if isinstance(appendix, list)
        else None,
    }

    table_summaries = []
    column_inventory = []
    table_columns = {}

    # Inspect each table separately
    for table_name, rows in data.items():

        if not isinstance(rows, list):
            print(f"WARNING: {table_name} is not a list.")
            continue

        valid_rows = [
            row for row in rows
            if isinstance(row, dict)
        ]

        invalid_rows = len(rows) - len(valid_rows)

        if not valid_rows:
            table_summaries.append({
                "fiscal_year": year,
                "table_name": table_name,
                "row_count": len(rows),
                "valid_dictionary_rows": 0,
                "invalid_rows": invalid_rows,
                "first_row_column_count": 0,
                "observed_column_count": 0,
                "rows_with_schema_variation": 0,
            })
            table_columns[table_name] = []
            continue

        # Preserve the column order in the first record
        first_columns = list(valid_rows[0].keys())
        first_column_set = set(first_columns)

        # Find every column observed in any record
        all_columns = list(first_columns)
        seen_columns = set(first_columns)

        schema_variation_count = 0

        for row in valid_rows:

            row_keys = set(row.keys())

            if row_keys != first_column_set:
                schema_variation_count += 1

            for column in row.keys():
                if column not in seen_columns:
                    all_columns.append(column)
                    seen_columns.add(column)

        table_columns[table_name] = all_columns

        table_summaries.append({
            "fiscal_year": year,
            "table_name": table_name,
            "row_count": len(rows),
            "valid_dictionary_rows": len(valid_rows),
            "invalid_rows": invalid_rows,
            "first_row_column_count": len(first_columns),
            "observed_column_count": len(all_columns),
            "rows_with_schema_variation": schema_variation_count,
        })

        # Infer value types using up to the first 1,000 records.
        # This is an initial inspection, not a complete type audit.
        sample_rows = valid_rows[:1000]

        for position, column in enumerate(all_columns, start=1):

            types = Counter()

            for row in sample_rows:
                if column in row:
                    value = row[column]

                    if value is None:
                        label = "null"
                    elif isinstance(value, bool):
                        label = "boolean"
                    elif isinstance(value, int):
                        label = "integer"
                    elif isinstance(value, float):
                        label = "float"
                    elif isinstance(value, str):
                        label = "string"
                    elif isinstance(value, list):
                        label = "list"
                    elif isinstance(value, dict):
                        label = "object"
                    else:
                        label = type(value).__name__

                    types[label] += 1

            column_inventory.append({
                "fiscal_year": year,
                "table_name": table_name,
                "column_position": position,
                "column_name": column,
                "sample_value_types": "; ".join(
                    f"{name}: {count}"
                    for name, count in sorted(types.items())
                ),
                "type_sample_size": len(sample_rows),
            })

    # Release the parsed document before moving to the next year
    del document
    gc.collect()

    print(f"Metadata saved: {metadata_path}")
    print("Tables found:", list(table_columns.keys()))

    return (
        metadata_summary,
        table_summaries,
        column_inventory,
        table_columns,
    )

In [ ]:
# CELL 3: Run the inspection across all years

metadata_results = []
table_results = []
column_results = []

# Store column names for cross-year comparisons
schemas_by_year = {}

for year, zip_path in ZIP_PATHS.items():

    metadata, tables, columns, schemas = inspect_archive(
        year,
        zip_path
    )

    metadata_results.append(metadata)
    table_results.extend(tables)
    column_results.extend(columns)

    schemas_by_year[year] = schemas

# Convert inspection results to DataFrames
metadata_df = pd.DataFrame(metadata_results)
table_summary_df = pd.DataFrame(table_results)
column_inventory_df = pd.DataFrame(column_results)

# Save the reports
metadata_df.to_csv(
    REPORT_DIR / "metadata_summary.csv",
    index=False
)

table_summary_df.to_csv(
    REPORT_DIR / "table_schema_summary.csv",
    index=False
)

column_inventory_df.to_csv(
    REPORT_DIR / "column_inventory.csv",
    index=False
)

print("\n" + "=" * 65)
print("TABLE SUMMARY")
print("=" * 65)
display(table_summary_df)

print("\n" + "=" * 65)
print("METADATA SUMMARY")
print("=" * 65)
display(
    metadata_df[
        [
            "fiscal_year",
            "data_product_name",
            "source",
            "creation_date",
            "data_table_names",
        ]
    ]
)

print("\n" + "=" * 65)
print("SAMPLE OF COLUMN INVENTORY")
print("=" * 65)
display(column_inventory_df.head(20))

print("\nInitial inspection reports saved to:")
print(REPORT_DIR)


Inspecting FY2022: raw_poultry_sampling_data_fy2022.zip
Metadata saved: /content/drive/MyDrive/raw_poultry_ml/metadata/metadata_FY2022.json
Tables found: ['primary_table_data', 'secondary_table_data']

Inspecting FY2023: raw_poultry_sampling_data_fy2023.zip
Metadata saved: /content/drive/MyDrive/raw_poultry_ml/metadata/metadata_FY2023.json
Tables found: ['primary_table_data', 'secondary_table_data']

Inspecting FY2024: raw_poultry_sampling_data_fy2024.zip
Metadata saved: /content/drive/MyDrive/raw_poultry_ml/metadata/metadata_FY2024.json
Tables found: ['primary_table_data', 'secondary_table_data']

Inspecting FY2025: raw_poultry_sampling_data_fy2025.zip
Metadata saved: /content/drive/MyDrive/raw_poultry_ml/metadata/metadata_FY2025.json
Tables found: ['primary_table_data', 'secondary_table_data']

Inspecting FY2026: raw_poultry_sampling_data_fy2026.zip
Metadata saved: /content/drive/MyDrive/raw_poultry_ml/metadata/metadata_FY2026.json
Tables found: ['primary_table_data', 'secondary_tab

,fiscal_year,table_name,row_count,valid_dictionary_rows,invalid_rows,first_row_column_count,observed_column_count,rows_with_schema_variation
0,2022,primary_table_data,34613,34613,0,30,30,0
1,2022,secondary_table_data,9331,9331,0,141,141,0
2,2023,primary_table_data,32035,32035,0,30,30,0
3,2023,secondary_table_data,9218,9218,0,141,141,0
4,2024,primary_table_data,28941,28941,0,30,30,0
5,2024,secondary_table_data,5206,5206,0,141,141,0
6,2025,primary_table_data,28564,28564,0,30,30,0
7,2025,secondary_table_data,5839,5839,0,141,141,0
8,2026,primary_table_data,21102,21102,0,30,30,0
9,2026,secondary_table_data,4219,4219,0,141,141,0



METADATA SUMMARY


,fiscal_year,data_product_name,source,creation_date,data_table_names
0,2022,Raw Poultry Product Sampling - FY2022,PHIS,"March 19, 2026",primary_table_data; secondary_table_data
1,2023,Raw Poultry Product Sampling - FY2023,PHIS,"March 19, 2026",primary_table_data; secondary_table_data
2,2024,Raw Poultry Product Sampling - FY2024,PHIS,"March 19, 2026",primary_table_data; secondary_table_data
3,2025,Raw Poultry Product Sampling - FY2025,PHIS,"March 19, 2026",primary_table_data; secondary_table_data
4,2026,Raw Poultry Product Sampling - FY2026,PHIS,"September 23, 2026",primary_table_data; secondary_table_data



SAMPLE OF COLUMN INVENTORY


,fiscal_year,table_name,column_position,column_name,sample_value_types,type_sample_size
0,2022,primary_table_data,1,form_id,string: 1000,1000
1,2022,primary_table_data,2,sample_number,integer: 1000,1000
2,2022,primary_table_data,3,establishment_id,integer: 1000,1000
3,2022,primary_table_data,4,establishment_number,string: 1000,1000
4,2022,primary_table_data,5,establishment_name,string: 1000,1000
5,2022,primary_table_data,6,establishment_state,string: 1000,1000
6,2022,primary_table_data,7,project_code,string: 1000,1000
7,2022,primary_table_data,8,project_name,string: 1000,1000
8,2022,primary_table_data,9,collection_date,string: 1000,1000
9,2022,primary_table_data,10,sample_source_name,string: 1000,1000



Initial inspection reports saved to:
/content/drive/MyDrive/raw_poultry_ml/reports


In [ ]:
# CELL 4: Compare schemas across fiscal years

comparison_results = []

# Use FY2023 as the reference schema
reference_year = 2022
reference_schemas = schemas_by_year[reference_year]

for table_name, reference_columns in reference_schemas.items():

    reference_set = set(reference_columns)

    for year in YEARS:

        current_columns = schemas_by_year[year].get(
            table_name, []
        )

        current_set = set(current_columns)

        comparison_results.append({
            "fiscal_year": year,
            "table_name": table_name,
            "reference_year": reference_year,
            "reference_column_count": len(reference_columns),
            "current_column_count": len(current_columns),
            "same_columns": current_set == reference_set,
            "same_order": current_columns == reference_columns,
            "missing_columns": "; ".join(
                sorted(reference_set - current_set)
            ),
            "additional_columns": "; ".join(
                sorted(current_set - reference_set)
            ),
        })

schema_comparison_df = pd.DataFrame(comparison_results)

schema_comparison_df.to_csv(
    REPORT_DIR / "schema_comparison.csv",
    index=False
)

display(schema_comparison_df)

if schema_comparison_df["same_columns"].all():
    print("\nPASS: All inspected tables have the same column sets.")
else:
    print("\nWARNING: Some column schemas differ. Review the report.")

print("\nAll reports are located in:", REPORT_DIR)

,fiscal_year,table_name,reference_year,reference_column_count,current_column_count,same_columns,same_order,missing_columns,additional_columns
0,2022,primary_table_data,2022,30,30,True,True,,
1,2023,primary_table_data,2022,30,30,True,True,,
2,2024,primary_table_data,2022,30,30,True,True,,
3,2025,primary_table_data,2022,30,30,True,True,,
4,2026,primary_table_data,2022,30,30,True,True,,
5,2022,secondary_table_data,2022,141,141,True,True,,
6,2023,secondary_table_data,2022,141,141,True,True,,
7,2024,secondary_table_data,2022,141,141,True,True,,
8,2025,secondary_table_data,2022,141,141,True,True,,
9,2026,secondary_table_data,2022,141,141,True,True,,



PASS: All inspected tables have the same column sets.

All reports are located in: /content/drive/MyDrive/raw_poultry_ml/reports


# Clean and Combine

In [ ]:

# CELL 1: Project setup


# Main project folder
PROJECT_DIR = Path(
    "/content/drive/MyDrive/raw_poultry_ml"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"
METADATA_DIR = PROJECT_DIR / "metadata"

# Create necessary folders
for folder in [
    INTERIM_DIR / "primary",
    INTERIM_DIR / "secondary",
    PROCESSED_DIR,
    REPORT_DIR,
    METADATA_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

YEARS = [2022, 2023, 2024, 2025, 2026]

ZIP_PATHS = {
    year: RAW_DIR / (
        f"raw_poultry_sampling_data_fy{year}.zip"
    )
    for year in YEARS
}

# Verify that all files exist
missing_files = [
    str(path)
    for path in ZIP_PATHS.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing ZIP files:\n" + "\n".join(missing_files)
    )

print("Project setup successful.")
print("Years:", YEARS)
print("Raw folder:", RAW_DIR)

for year, path in ZIP_PATHS.items():
    print(f"FY{year}: {path.name} — FOUND")


Project setup successful.
Years: [2022, 2023, 2024, 2025, 2026]
Raw folder: /content/drive/MyDrive/raw_poultry_ml/data/raw
FY2022: raw_poultry_sampling_data_fy2022.zip — FOUND
FY2023: raw_poultry_sampling_data_fy2023.zip — FOUND
FY2024: raw_poultry_sampling_data_fy2024.zip — FOUND
FY2025: raw_poultry_sampling_data_fy2025.zip — FOUND
FY2026: raw_poultry_sampling_data_fy2026.zip — FOUND


In [ ]:

# CELL 2: Extract and prepare yearly datasets

def read_json_archive(zip_path):
    """Read the JSON document inside a ZIP archive."""

    with zipfile.ZipFile(zip_path, "r") as archive:
        json_files = [
            name for name in archive.namelist()
            if name.lower().endswith(".json")
        ]

        if len(json_files) != 1:
            raise ValueError(
                f"Expected one JSON file in {zip_path.name}; "
                f"found {json_files}"
            )

        with archive.open(json_files[0], "r") as file:
            document = json.load(file)

    # Handle a one-element list at the JSON root
    if isinstance(document, list):
        if len(document) != 1:
            raise ValueError(
                f"Unexpected JSON root list in {zip_path.name}"
            )
        document = document[0]

    if not isinstance(document, dict):
        raise ValueError("JSON root must be an object.")

    data = document.get("data")

    if not isinstance(data, dict):
        raise ValueError(
            f"No valid 'data' object in {zip_path.name}"
        )

    return document, data


def find_table(data, keyword):
    """Find the primary or secondary table by its name."""

    matches = [
        key for key in data
        if keyword in key.lower()
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one '{keyword}' table; found {matches}"
        )

    table_name = matches[0]
    records = data[table_name]

    if not isinstance(records, list):
        raise ValueError(
            f"{table_name} is not a list of records."
        )

    if not all(isinstance(row, dict) for row in records):
        raise ValueError(
            f"{table_name} contains non-dictionary rows. "
            "Inspect these records before continuing."
        )

    return table_name, records


def standardize_missing_values(df):
    """
    Normalize obvious missing placeholders.
    Do not convert measurement columns to numeric yet.
    """

    df = df.copy()

    for column in df.columns:
        if (
            pd.api.types.is_object_dtype(df[column])
            or isinstance(df[column].dtype, pd.StringDtype)
        ):
            def clean_value(value):
                if isinstance(value, str):
                    value = value.strip()

                    if value.upper() == "NULL" or value == "":
                        return pd.NA

                    return value

                return value

            df[column] = df[column].map(clean_value)

    return df


# Reports and schema tracking
import_summaries = []
missingness_records = []
schema_inventory = {}

for year, zip_path in ZIP_PATHS.items():

    print(f"\nProcessing FY{year}...")

    document, data = read_json_archive(zip_path)

    # Process the two tables separately
    for table_type in ["primary", "secondary"]:

        table_name, records = find_table(data, table_type)

        df = pd.DataFrame(records)

        # Preserve the input schema before adding project columns
        original_columns = list(df.columns)
        schema_inventory[(year, table_type)] = original_columns

        # Count repeated full rows before adding provenance columns.
        # This is a diagnostic only; no rows are deleted.
        duplicate_count = int(df.duplicated().sum())

        # Normalize obvious missing placeholders
        df = standardize_missing_values(df)

        # Add traceability columns
        reserved_columns = {
            "fiscal_year",
            "source_archive",
            "source_row_number",
        }

        conflicts = reserved_columns.intersection(df.columns)

        if conflicts:
            raise ValueError(
                f"FY{year} {table_name} already contains "
                f"reserved columns: {sorted(conflicts)}. "
                "Rename these source columns deliberately first."
            )

        df["fiscal_year"] = year
        df["source_archive"] = zip_path.name
        df["source_row_number"] = range(1, len(df) + 1)

        # Save one standardized table per year
        output_dir = INTERIM_DIR / table_type
        output_path = output_dir / (
            f"{table_type}_FY{year}.parquet"
        )

        df.to_parquet(output_path, index=False)

        # Record import diagnostics
        import_summaries.append({
            "fiscal_year": year,
            "table_type": table_type,
            "source_table_name": table_name,
            "row_count": len(df),
            "source_column_count": len(original_columns),
            "exact_duplicate_rows_before_provenance":
                duplicate_count,
            "saved_file": str(output_path),
        })

        # Missing-value report for source columns
        for column in original_columns:
            missing_count = int(df[column].isna().sum())

            missingness_records.append({
                "fiscal_year": year,
                "table_type": table_type,
                "column_name": column,
                "missing_count": missing_count,
                "missing_pct": (
                    100 * missing_count / len(df)
                    if len(df) > 0 else 0
                ),
            })

        print(
            f"  {table_type}: {len(df):,} rows, "
            f"{len(original_columns)} source columns"
        )
        print(f"  Saved: {output_path.name}")

        del df
        gc.collect()

    # Release each year's parsed JSON before reading the next archive
    del document, data
    gc.collect()

# Save reports
import_summary_df = pd.DataFrame(import_summaries)
missingness_df = pd.DataFrame(missingness_records)

import_summary_df.to_csv(
    REPORT_DIR / "import_summary.csv",
    index=False
)

missingness_df.to_csv(
    REPORT_DIR / "missingness_by_year.csv",
    index=False
)

print("\nYEARLY EXTRACTION COMPLETED")
display(import_summary_df)



Processing FY2022...
  primary: 34,613 rows, 30 source columns
  Saved: primary_FY2022.parquet


/tmp/ipykernel_626/1335369069.py:146: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fiscal_year"] = year
/tmp/ipykernel_626/1335369069.py:147: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["source_archive"] = zip_path.name
/tmp/ipykernel_626/1335369069.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

  secondary: 9,331 rows, 141 source columns
  Saved: secondary_FY2022.parquet

Processing FY2023...
  primary: 32,035 rows, 30 source columns
  Saved: primary_FY2023.parquet


/tmp/ipykernel_626/1335369069.py:146: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fiscal_year"] = year
/tmp/ipykernel_626/1335369069.py:147: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["source_archive"] = zip_path.name
/tmp/ipykernel_626/1335369069.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

  secondary: 9,218 rows, 141 source columns
  Saved: secondary_FY2023.parquet

Processing FY2024...
  primary: 28,941 rows, 30 source columns
  Saved: primary_FY2024.parquet


...

/tmp/ipykernel_626/1335369069.py:146: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fiscal_year"] = year
/tmp/ipykernel_626/1335369069.py:147: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["source_archive"] = zip_path.name
/tmp/ipykernel_626/1335369069.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

  secondary: 5,839 rows, 141 source columns
  Saved: secondary_FY2025.parquet

Processing FY2026...
  primary: 21,102 rows, 30 source columns
  Saved: primary_FY2026.parquet


/tmp/ipykernel_626/1335369069.py:146: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fiscal_year"] = year
/tmp/ipykernel_626/1335369069.py:147: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["source_archive"] = zip_path.name
/tmp/ipykernel_626/1335369069.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

  secondary: 4,219 rows, 141 source columns
  Saved: secondary_FY2026.parquet

YEARLY EXTRACTION COMPLETED


,fiscal_year,table_type,source_table_name,row_count,source_column_count,exact_duplicate_rows_before_provenance,saved_file
0,2022,primary,primary_table_data,34613,30,0,/content/drive/MyDrive/raw_poultry_ml/data/int...
1,2022,secondary,secondary_table_data,9331,141,0,/content/drive/MyDrive/raw_poultry_ml/data/int...
2,2023,primary,primary_table_data,32035,30,0,/content/drive/MyDrive/raw_poultry_ml/data/int...
3,2023,secondary,secondary_table_data,9218,141,0,/content/drive/MyDrive/raw_poultry_ml/data/int...
4,2024,primary,primary_table_data,28941,30,0,/content/drive/MyDrive/raw_poultry_ml/data/int...
5,2024,secondary,secondary_table_data,5206,141,0,/content/drive/MyDrive/raw_poultry_ml/data/int...
6,2025,primary,primary_table_data,28564,30,0,/content/drive/MyDrive/raw_poultry_ml/data/int...
7,2025,secondary,secondary_table_data,5839,141,0,/content/drive/MyDrive/raw_poultry_ml/data/int...
8,2026,primary,primary_table_data,21102,30,0,/content/drive/MyDrive/raw_poultry_ml/data/int...
9,2026,secondary,secondary_table_data,4219,141,0,/content/drive/MyDrive/raw_poultry_ml/data/int...


In [ ]:

# CELL 3: Schema comparison and combined datasets

schema_records = []

for table_type in ["primary", "secondary"]:

    reference_columns = schema_inventory.get(
        (2022, table_type)
    )

    if reference_columns is None:
        raise ValueError(
            f"FY2022 {table_type} schema is missing."
        )

    reference_set = set(reference_columns)

    for year in YEARS:
        current_columns = schema_inventory.get(
            (year, table_type)
        )

        if current_columns is None:
            raise ValueError(
                f"Missing schema for FY{year} {table_type}."
            )

        current_set = set(current_columns)

        schema_records.append({
            "fiscal_year": year,
            "table_type": table_type,
            "reference_year": 2022,
            "reference_column_count": len(reference_columns),
            "current_column_count": len(current_columns),
            "same_columns": current_set == reference_set,
            "same_column_order":
                current_columns == reference_columns,
            "missing_columns_vs_FY2022": "; ".join(
                sorted(reference_set - current_set)
            ),
            "additional_columns_vs_FY2022": "; ".join(
                sorted(current_set - reference_set)
            ),
        })

schema_comparison_df = pd.DataFrame(schema_records)

schema_comparison_df.to_csv(
    REPORT_DIR / "schema_comparison.csv",
    index=False
)

print("SCHEMA COMPARISON")
display(schema_comparison_df)


# Combine primary and secondary tables separately
combined_summaries = []

for table_type in ["primary", "secondary"]:

    yearly_frames = []

    for year in YEARS:
        file_path = (
            INTERIM_DIR / table_type
            / f"{table_type}_FY{year}.parquet"
        )

        if not file_path.exists():
            raise FileNotFoundError(
                f"Missing interim file: {file_path}"
            )

        yearly_frames.append(pd.read_parquet(file_path))

    # Combine records vertically; preserve all columns
    combined_df = pd.concat(
        yearly_frames,
        ignore_index=True,
        sort=False
    )

    output_path = (
        PROCESSED_DIR / f"{table_type}_all_years.parquet"
    )

    combined_df.to_parquet(output_path, index=False)

    combined_summaries.append({
        "table_type": table_type,
        "total_rows": len(combined_df),
        "total_columns_including_provenance":
            len(combined_df.columns),
        "fiscal_years": ", ".join(
            str(y) for y in sorted(
                combined_df["fiscal_year"].dropna().unique()
            )
        ),
        "output_file": str(output_path),
    })

    print(
        f"\nCombined {table_type}: "
        f"{len(combined_df):,} rows, "
        f"{len(combined_df.columns)} columns"
    )
    print(f"Saved to: {output_path}")

    del yearly_frames, combined_df
    gc.collect()


combined_summary_df = pd.DataFrame(combined_summaries)

combined_summary_df.to_csv(
    REPORT_DIR / "combined_dataset_summary.csv",
    index=False
)

print("\nCOMBINED DATASET SUMMARY")
display(combined_summary_df)

print("\nReports saved in:", REPORT_DIR)


SCHEMA COMPARISON


,fiscal_year,table_type,reference_year,reference_column_count,current_column_count,same_columns,same_column_order,missing_columns_vs_FY2022,additional_columns_vs_FY2022
0,2022,primary,2022,30,30,True,True,,
1,2023,primary,2022,30,30,True,True,,
2,2024,primary,2022,30,30,True,True,,
3,2025,primary,2022,30,30,True,True,,
4,2026,primary,2022,30,30,True,True,,
5,2022,secondary,2022,141,141,True,True,,
6,2023,secondary,2022,141,141,True,True,,
7,2024,secondary,2022,141,141,True,True,,
8,2025,secondary,2022,141,141,True,True,,
9,2026,secondary,2022,141,141,True,True,,



Combined primary: 145,255 rows, 33 columns
Saved to: /content/drive/MyDrive/raw_poultry_ml/data/processed/primary_all_years.parquet


/tmp/ipykernel_626/15705442.py:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(



Combined secondary: 33,813 rows, 144 columns
Saved to: /content/drive/MyDrive/raw_poultry_ml/data/processed/secondary_all_years.parquet

COMBINED DATASET SUMMARY


,table_type,total_rows,total_columns_including_provenance,fiscal_years,output_file
0,primary,145255,33,"2022, 2023, 2024, 2025, 2026",/content/drive/MyDrive/raw_poultry_ml/data/pro...
1,secondary,33813,144,"2022, 2023, 2024, 2025, 2026",/content/drive/MyDrive/raw_poultry_ml/data/pro...



Reports saved in: /content/drive/MyDrive/raw_poultry_ml/reports


# Data quality

In [ ]:

# CELL 1: Load combined datasets


PROJECT_DIR = Path(
    "/content/drive/MyDrive/raw_poultry_ml"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

primary_path = PROCESSED_DIR / "primary_all_years.parquet"
secondary_path = PROCESSED_DIR / "secondary_all_years.parquet"

primary_df = pd.read_parquet(primary_path)
secondary_df = pd.read_parquet(secondary_path)

print("DATA LOADED SUCCESSFULLY")
print("-" * 50)

print("Primary shape:", primary_df.shape)
print("Secondary shape:", secondary_df.shape)

print("\nPrimary records by fiscal year:")
display(primary_df["fiscal_year"].value_counts().sort_index())

print("\nSecondary records by fiscal year:")
display(secondary_df["fiscal_year"].value_counts().sort_index())

print("\nPrimary columns:")
print(primary_df.columns.tolist())

print("\nSecondary columns:")
print(secondary_df.columns.tolist())


DATA LOADED SUCCESSFULLY
--------------------------------------------------
Primary shape: (145255, 33)
Secondary shape: (33813, 144)

Primary records by fiscal year:


,count
fiscal_year,
2022,34613
2023,32035
2024,28941
2025,28564
2026,21102



Secondary records by fiscal year:


,count
fiscal_year,
2022,9331
2023,9218
2024,5206
2025,5839
2026,4219



Primary columns:
['form_id', 'sample_number', 'establishment_id', 'establishment_number', 'establishment_name', 'establishment_state', 'project_code', 'project_name', 'collection_date', 'sample_source_name', 'product_type', 'not_analyzed', 'not_analyzed_explain', 'salmonella_sp_analysis', 'salmonella_sp_analysis_canceled', 'salmonellae_mpn_g', 'salmonellae_mpn_g_canceled', 'salmonellae_mpn_combination', 'salmonella_quantitative_cfu_g', 'salmonella_quantitative_cfu_ml', 'salmonella_mpn_qualitative', 'campylobacter_analysis_1ml', 'campylobacter_analysis_1ml_canceled', 'campylobacter_analysis_30ml', 'campylobacter_analysis_30ml_canceled', 'aerobic_plate_count_mpn_ml', 'enterobacteriaceae_mpn_ml', 'aerobic_plate_count_mpn_100_sq_cm', 'enterobacteriaceae_mpn_100_sq_cm', 'aerobic_plate_count_mpn_g', 'fiscal_year', 'source_archive', 'source_row_number']

Secondary columns:
['form_id', 'secondary_sample_number', 'tertiary_sample_number', 'pathogen_number', 'fsis_number', 'salmonella_allele_co

In [ ]:

# CELL 2: Data quality audit

def audit_dataset(df, table_name):
    # Exclude tracking fields from source-data quality summaries
    excluded = {"source_archive", "source_row_number"}

    source_columns = [
        col for col in df.columns
        if col not in excluded
    ]

    # Missing values by column and fiscal year
    missing_records = []

    for year, group in df.groupby("fiscal_year", dropna=False):
        for col in source_columns:
            missing_count = int(group[col].isna().sum())

            missing_records.append({
                "table": table_name,
                "fiscal_year": year,
                "column": col,
                "missing_count": missing_count,
                "total_rows": len(group),
                "missing_pct": (
                    missing_count / len(group) * 100
                    if len(group) else 0
                )
            })

    missing_df = pd.DataFrame(missing_records)

    # Data types and overall missingness
    column_records = []

    for col in source_columns:
        column_records.append({
            "table": table_name,
            "column": col,
            "dtype": str(df[col].dtype),
            "missing_count": int(df[col].isna().sum()),
            "missing_pct": (
                df[col].isna().mean() * 100
            ),
            "unique_nonmissing_values": int(
                df[col].nunique(dropna=True)
            )
        })

    column_df = pd.DataFrame(column_records)

    # Count exact duplicate source rows within each fiscal year.
    # Duplicate-looking rows are flagged, not removed.
    duplicate_records = []

    duplicate_columns = [
        col for col in df.columns
        if col not in {"source_archive", "source_row_number"}
    ]

    for year, group in df.groupby("fiscal_year", dropna=False):
        duplicate_count = int(
            group.duplicated(
                subset=duplicate_columns
            ).sum()
        )

        duplicate_records.append({
            "table": table_name,
            "fiscal_year": year,
            "rows": len(group),
            "exact_duplicate_rows": duplicate_count
        })

    duplicates_df = pd.DataFrame(duplicate_records)

    return missing_df, column_df, duplicates_df


primary_missing, primary_columns, primary_duplicates = (
    audit_dataset(primary_df, "primary")
)

secondary_missing, secondary_columns, secondary_duplicates = (
    audit_dataset(secondary_df, "secondary")
)

# Save reports
pd.concat(
    [primary_missing, secondary_missing],
    ignore_index=True
).to_csv(REPORT_DIR / "missingness_audit.csv", index=False)

pd.concat(
    [primary_columns, secondary_columns],
    ignore_index=True
).to_csv(REPORT_DIR / "column_quality_audit.csv", index=False)

pd.concat(
    [primary_duplicates, secondary_duplicates],
    ignore_index=True
).to_csv(REPORT_DIR / "duplicate_audit.csv", index=False)

print("PRIMARY: highest missingness")
display(
    primary_columns.sort_values(
        "missing_pct", ascending=False
    ).head(15)
)

print("\nPRIMARY: duplicate audit")
display(primary_duplicates)

print("\nSECONDARY: duplicate audit")
display(secondary_duplicates)

print("\nReports saved to:", REPORT_DIR)


PRIMARY: highest missingness


,table,column,dtype,missing_count,missing_pct,unique_nonmissing_values
27,primary,aerobic_plate_count_mpn_100_sq_cm,object,145255,100.000000,0
21,primary,campylobacter_analysis_1ml,object,145255,100.000000,0
22,primary,campylobacter_analysis_1ml_canceled,object,145255,100.000000,0
20,primary,salmonella_mpn_qualitative,object,145255,100.000000,0
17,primary,salmonellae_mpn_combination,object,145255,100.000000,0
16,primary,salmonellae_mpn_g_canceled,object,145255,100.000000,0
15,primary,salmonellae_mpn_g,object,145255,100.000000,0
28,primary,enterobacteriaceae_mpn_100_sq_cm,object,145255,100.000000,0
14,primary,salmonella_sp_analysis_canceled,object,145221,99.976593,8
12,primary,not_analyzed_explain,object,145217,99.973839,19



PRIMARY: duplicate audit


,table,fiscal_year,rows,exact_duplicate_rows
0,primary,2022,34613,0
1,primary,2023,32035,0
2,primary,2024,28941,0
3,primary,2025,28564,0
4,primary,2026,21102,0



SECONDARY: duplicate audit


,table,fiscal_year,rows,exact_duplicate_rows
0,secondary,2022,9331,0
1,secondary,2023,9218,0
2,secondary,2024,5206,0
3,secondary,2025,5839,0
4,secondary,2026,4219,0



Reports saved to: /content/drive/MyDrive/raw_poultry_ml/reports


In [ ]:

# CELL 3: Inspect pathogen outcome categories

print("Possible pathogen analysis columns:")
pathogen_columns = [
    col for col in primary_df.columns
    if any(
        pathogen in col.lower()
        for pathogen in ["salmonella", "campylobacter"]
    )
    and "analysis" in col.lower()
]

print(pathogen_columns)

distribution_records = []

for col in pathogen_columns:
    print(f"\nResult distribution: {col}")

    counts = (
        primary_df.groupby(
            ["fiscal_year", col],
            dropna=False
        )
        .size()
        .reset_index(name="count")
    )

    counts["column"] = col

    display(counts)

    for _, row in counts.iterrows():
        distribution_records.append({
            "column": col,
            "fiscal_year": row["fiscal_year"],
            "result": (
                "<MISSING>"
                if pd.isna(row[col])
                else str(row[col])
            ),
            "count": row["count"]
        })

distribution_df = pd.DataFrame(distribution_records)

distribution_df.to_csv(
    REPORT_DIR / "pathogen_result_distributions.csv",
    index=False
)

print("\nResult distribution report saved.")


Possible pathogen analysis columns:
['salmonella_sp_analysis', 'salmonella_sp_analysis_canceled', 'campylobacter_analysis_1ml', 'campylobacter_analysis_1ml_canceled', 'campylobacter_analysis_30ml', 'campylobacter_analysis_30ml_canceled']

Result distribution: salmonella_sp_analysis


,fiscal_year,salmonella_sp_analysis,count,column
0,2022,Negative,29708,salmonella_sp_analysis
1,2022,Positive,4902,salmonella_sp_analysis
2,2022,NaN,3,salmonella_sp_analysis
3,2023,Negative,28381,salmonella_sp_analysis
4,2023,Positive,3639,salmonella_sp_analysis
5,2023,NaN,15,salmonella_sp_analysis
6,2024,Negative,26429,salmonella_sp_analysis
7,2024,Positive,2476,salmonella_sp_analysis
8,2024,NaN,36,salmonella_sp_analysis
9,2025,Negative,25756,salmonella_sp_analysis



Result distribution: salmonella_sp_analysis_canceled


,fiscal_year,salmonella_sp_analysis_canceled,count,column
0,2022,25-12 Lab problem - spilled,1,salmonella_sp_analysis_canceled
1,2022,25-20 Other - (Describe on form in up to 54 c...,1,salmonella_sp_analysis_canceled
2,2022,25-7 Lab problem - sample inadvertently disca...,1,salmonella_sp_analysis_canceled
3,2022,NaN,34610,salmonella_sp_analysis_canceled
4,2023,25 Laboratory Problem,1,salmonella_sp_analysis_canceled
5,2023,25-18 Lab problem - equipment failure,14,salmonella_sp_analysis_canceled
6,2023,NaN,32020,salmonella_sp_analysis_canceled
7,2024,25 Laboratory Problem,2,salmonella_sp_analysis_canceled
8,2024,25-18 Lab problem - equipment failure,1,salmonella_sp_analysis_canceled
9,2024,25-9 Lab problem - sample logged-in/analyzed ...,1,salmonella_sp_analysis_canceled



Result distribution: campylobacter_analysis_1ml


...


Result distribution: campylobacter_analysis_30ml


,fiscal_year,campylobacter_analysis_30ml,count,column
0,2022,Negative,24504,campylobacter_analysis_30ml
1,2022,Positive,4436,campylobacter_analysis_30ml
2,2022,NaN,5673,campylobacter_analysis_30ml
3,2023,Negative,23624,campylobacter_analysis_30ml
4,2023,Positive,5587,campylobacter_analysis_30ml
5,2023,NaN,2824,campylobacter_analysis_30ml
6,2024,Negative,11664,campylobacter_analysis_30ml
7,2024,Positive,2734,campylobacter_analysis_30ml
8,2024,NaN,14543,campylobacter_analysis_30ml
9,2025,Negative,11845,campylobacter_analysis_30ml



Result distribution: campylobacter_analysis_30ml_canceled


,fiscal_year,campylobacter_analysis_30ml_canceled,count,column
0,2022,13 Insufficient tissue or sample,1,campylobacter_analysis_30ml_canceled
1,2022,25-12 Lab problem - spilled,1,campylobacter_analysis_30ml_canceled
2,2022,25-20 Other - (Describe on form in up to 54 c...,1,campylobacter_analysis_30ml_canceled
3,2022,46 Temperature too low,17,campylobacter_analysis_30ml_canceled
4,2022,48 Headquarters/PDD/DO/OIA Discard,1,campylobacter_analysis_30ml_canceled
5,2022,NaN,34592,campylobacter_analysis_30ml_canceled
6,2023,13 Insufficient tissue or sample,30,campylobacter_analysis_30ml_canceled
7,2023,25 Laboratory Problem,15,campylobacter_analysis_30ml_canceled
8,2023,25-10 Lab problem - sample lost in lab,1,campylobacter_analysis_30ml_canceled
9,2023,25-18 Lab problem - equipment failure,7,campylobacter_analysis_30ml_canceled



Result distribution report saved.


In [ ]:

# NEXT CELL: Inspect candidate predictors for Salmonella ML

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_DIR = Path(
    "/content/drive/MyDrive/raw_poultry_ml"
)

primary_path = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "primary_all_years.parquet"
)

primary_df = pd.read_parquet(primary_path)

# Candidate columns are selected by name for an initial review.
# This does not automatically make them approved ML features.
candidate_keywords = [
    "date",
    "state",
    "establishment",
    "project",
    "sample_source",
    "product_type",
    "product_class",
    "salmonella_sp_analysis",
]

candidate_columns = [
    col for col in primary_df.columns
    if any(
        keyword in col.lower()
        for keyword in candidate_keywords
    )
]

feature_audit = pd.DataFrame({
    "column": candidate_columns,
    "dtype": [
        str(primary_df[col].dtype)
        for col in candidate_columns
    ],
    "missing_count": [
        int(primary_df[col].isna().sum())
        for col in candidate_columns
    ],
    "missing_pct": [
        round(primary_df[col].isna().mean() * 100, 2)
        for col in candidate_columns
    ],
    "unique_values": [
        int(primary_df[col].nunique(dropna=True))
        for col in candidate_columns
    ],
})

display(
    feature_audit.sort_values("missing_pct")
)

feature_audit.to_csv(
    PROJECT_DIR
    / "reports"
    / "candidate_feature_audit.csv",
    index=False
)

print("Saved candidate feature audit.")


,column,dtype,missing_count,missing_pct,unique_values
0,establishment_id,int64,0,0.00,941
1,establishment_number,object,0,0.00,947
2,establishment_name,object,0,0.00,808
3,establishment_state,object,0,0.00,46
4,project_code,object,0,0.00,14
5,project_name,object,0,0.00,14
6,collection_date,object,0,0.00,1470
7,sample_source_name,object,0,0.00,6
9,salmonella_sp_analysis,object,72,0.05,2
8,product_type,object,70735,48.70,8


Saved candidate feature audit.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# CELL 4: Create the first Salmonella modeling dataset

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/raw_poultry_ml"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load the complete primary dataset
primary_df = pd.read_parquet(
    PROCESSED_DIR / "primary_all_years.parquet"
)

# 1. Define the binary target
target_mapping = {
    "Negative": 0,
    "Positive": 1,
}

target_column = "salmonella_sp_analysis"

# Identify valid results; retain the original dataset unchanged
valid_mask = primary_df[target_column].isin(
    target_mapping.keys()
)

model_df = primary_df.loc[valid_mask].copy()

model_df["target_salmonella"] = (
    model_df[target_column].map(target_mapping).astype("int8")
)

# Report any other nonmissing result values
unrecognized = (
    primary_df.loc[
        primary_df[target_column].notna()
        & ~primary_df[target_column].isin(
            target_mapping.keys()
        ),
        target_column
    ]
    .value_counts()
)

print("Rows in original dataset:", len(primary_df))
print("Rows with valid target:", len(model_df))
print("Rows without valid target:", len(primary_df) - len(model_df))
print("\nUnrecognized nonmissing target values:")
print(unrecognized if not unrecognized.empty else "None")

# 2. Parse collection dates and create month
# Use mixed-format parsing when supported by pandas.
try:
    parsed_dates = pd.to_datetime(
        model_df["collection_date"],
        errors="coerce",
        format="mixed"
    )
except (TypeError, ValueError):
    parsed_dates = pd.to_datetime(
        model_df["collection_date"],
        errors="coerce"
    )

invalid_dates = int(parsed_dates.isna().sum())

model_df["collection_month"] = (
    parsed_dates.dt.month.astype("Int64")
)

print("\nDates that could not be parsed:", invalid_dates)

# 3. Select initial features plus audit/grouping metadata
feature_columns = [
    "establishment_state",
    "project_code",
    "sample_source_name",
    "product_type",
    "collection_month",
]

# These columns are retained for auditing and evaluation,
# not automatically used as model inputs.
metadata_columns = [
    "fiscal_year",
    "establishment_id",
    "collection_date",
    "source_archive",
    "source_row_number",
]

selected_columns = [
    col for col in feature_columns + metadata_columns
    if col in model_df.columns
]

selected_columns += [
    target_column,
    "target_salmonella",
]

salmonella_model_v1 = model_df[selected_columns].copy()

# 4. Save the modeling dataset
output_path = (
    PROCESSED_DIR / "salmonella_model_v1.parquet"
)

salmonella_model_v1.to_parquet(
    output_path,
    index=False
)

# Save target counts
target_summary = (
    salmonella_model_v1.groupby(
        ["fiscal_year", "target_salmonella"]
    )
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "negative", 1: "positive"})
)

target_summary.to_csv(
    REPORT_DIR / "salmonella_target_summary.csv"
)

print("\nMODELING DATASET CREATED")
print("Shape:", salmonella_model_v1.shape)
print("Saved to:", output_path)

print("\nTarget distribution:")
display(
    salmonella_model_v1["target_salmonella"]
    .value_counts()
    .rename(index={0: "Negative", 1: "Positive"})
    .to_frame("count")
)

print("\nTarget distribution by fiscal year:")
display(target_summary)

print("\nMissing values in initial features:")
display(
    salmonella_model_v1[feature_columns]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)


Rows in original dataset: 145255
Rows with valid target: 145183
Rows without valid target: 72

Unrecognized nonmissing target values:
None

Dates that could not be parsed: 0

MODELING DATASET CREATED
Shape: (145183, 12)
Saved to: /content/drive/MyDrive/raw_poultry_ml/data/processed/salmonella_model_v1.parquet

Target distribution:


,count
target_salmonella,
Negative,129327
Positive,15856



Target distribution by fiscal year:


target_salmonella,negative,positive
fiscal_year,,
2022,29708,4902
2023,28381,3639
2024,26429,2476
2025,25756,2798
2026,19053,2041



Missing values in initial features:


,missing_count
establishment_state,0
project_code,0
sample_source_name,0
product_type,70700
collection_month,0


In [ ]:

PROJECT_DIR = Path("/content/drive/MyDrive/raw_poultry_ml")

file_path = PROJECT_DIR / "data/processed/salmonella_model_v1.parquet"

df = pd.read_parquet(file_path)

display(df.head(10))

,establishment_state,project_code,sample_source_name,product_type,collection_month,fiscal_year,establishment_id,collection_date,source_archive,source_row_number,salmonella_sp_analysis,target_salmonella
0,GA,HC_CH_CARC01,Animal-Chicken-Broiler / Young Chicken Carcass...,None,10,2022,3,2021-10-01,raw_poultry_sampling_data_fy2022.zip,1,Negative,0
1,GA,HC_CPT_LBW01,Product-Raw-Intact-Chicken,Breast,10,2022,32,2021-10-01,raw_poultry_sampling_data_fy2022.zip,2,Negative,0
2,GA,HC_CPT_LBW01,Product-Raw-Intact-Chicken,Leg,10,2022,39,2021-10-01,raw_poultry_sampling_data_fy2022.zip,3,Positive,1
3,GA,HC_CH_COM01,"Product-Raw-Ground, Comminuted or Otherwise No...",None,10,2022,41,2021-10-01,raw_poultry_sampling_data_fy2022.zip,4,Positive,1
4,SC,HC_CH_CARC01,Animal-Chicken-Broiler / Young Chicken Carcass...,None,10,2022,564,2021-10-01,raw_poultry_sampling_data_fy2022.zip,5,Negative,0
5,NC,F_TU_COM01,"Product-Raw-Ground, Comminuted or Otherwise No...",None,10,2022,651,2021-10-01,raw_poultry_sampling_data_fy2022.zip,6,Positive,1
6,MA,HC_CH_COM01,"Product-Raw-Ground, Comminuted or Otherwise No...",None,10,2022,1831,2021-10-01,raw_poultry_sampling_data_fy2022.zip,7,Negative,0
7,WV,F_CPT_LBW01,Product-Raw-Intact-Chicken,Leg,10,2022,1906,2021-10-01,raw_poultry_sampling_data_fy2022.zip,8,Negative,0
8,VA,HC_CPT_LBW01,Product-Raw-Intact-Chicken,Wing,10,2022,2264,2021-10-01,raw_poultry_sampling_data_fy2022.zip,9,Negative,0
9,DE,HC_CPT_LBW01,Product-Raw-Intact-Chicken,Wing,10,2022,2267,2021-10-01,raw_poultry_sampling_data_fy2022.zip,10,Negative,0


In [ ]:
print("Sample source categories:")
display(
    df["sample_source_name"]
    .value_counts(dropna=False)
    .rename_axis("sample_source_name")
    .reset_index(name="count")
)

print("Product type categories:")
display(
    df["product_type"]
    .value_counts(dropna=False)
    .rename_axis("product_type")
    .reset_index(name="count")
)

Sample source categories:


,sample_source_name,count
0,Product-Raw-Intact-Chicken,67133
1,Animal-Chicken-Broiler / Young Chicken Carcass...,50810
2,"Product-Raw-Ground, Comminuted or Otherwise No...",15763
3,"Product-Raw-Ground, Comminuted or Otherwise No...",6353
4,Animal-Turkey-Turkey Carcass Sponge,5121
5,Animal-Chicken-Young Chicken,3


Product type categories:


,product_type,count
0,None,70700
1,Breast,34811
2,Leg,25443
3,Wing,14051
4,Quarter-Carcass,136
5,Half-Carcass,38
6,Whole Liver (only whole livers should be colle...,2
7,Gizzard,1
8,Neck,1
